# 06 — Hyperparameter Tuning with Optuna

**Goal:** Find the best hyperparameters for the winning model.

> ✅ Only **train** data. Test set still locked.

**Architecture:**
- Optuna study with TPE sampler
- Each trial → `TimeSeriesSplit` CV on train → returns CV RMSE
- Each trial → **nested MLflow run** inside a parent MLflow run
- After study: extract best params, visualize search, save params

> ⚠️ `nested=True` is REQUIRED when calling `mlflow.start_run()` inside another run.

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import optuna
import json
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial logs

from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin

try:
    from lightgbm import LGBMRegressor
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

from sklearn.ensemble import GradientBoostingRegressor

PROCESSED_PATH    = '../data/processed/'
MLFLOW_EXPERIMENT = 'energy-demand-prediction'
TARGET            = 'total load actual'
RANDOM_STATE      = 42
N_TRIALS          = 30   # increase to 100+ for production
TSCV              = TimeSeriesSplit(n_splits=5, gap=24)

d:\my_projects\Energy_demand_predictor\energy_demand\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data

In [2]:
train_raw = pd.read_csv(PROCESSED_PATH + 'train.csv')
train_raw['time'] = pd.to_datetime(train_raw['time'], utc=True)

with open(PROCESSED_PATH + 'feature_config.json') as f:
    config = json.load(f)

with open(PROCESSED_PATH + 'best_model.txt') as f:
    BEST_MODEL_NAME = f.read().strip()

ALL_FEATURES = config['all_features']
print(f'Tuning model: {BEST_MODEL_NAME}')
print(f'Train raw: {train_raw.shape}  |  Features: {len(ALL_FEATURES)}')

Tuning model: GradientBoosting
Train raw: (24544, 32)  |  Features: 40


## 2. Feature Transformer & Prepared Data

In [3]:
class EnergyFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lag_hours=None, add_cyclical=True):
        self.lag_hours=lag_hours or [1,24,48,168,336]; self.add_cyclical=add_cyclical
    def fit(self, X, y=None): return self
    def transform(self, X):
        df=X.copy(); df['time']=pd.to_datetime(df['time'],utc=True)
        df=df.sort_values('time').reset_index(drop=True)
        df=self._cal(df); df=self._cyc(df) if self.add_cyclical else df
        df=self._lag(df); df=self._roll(df); df=self._weather(df)
        return df.dropna().reset_index(drop=True)
    def _cal(self, df):
        t=df['time']; df['hour']=t.dt.hour; df['dow']=t.dt.dayofweek
        df['month']=t.dt.month; df['day_of_year']=t.dt.dayofyear
        df['week_of_year']=t.dt.isocalendar().week.astype(int)
        df['quarter']=t.dt.quarter; df['is_weekend']=(t.dt.dayofweek>=5).astype(int)
        df['season']=df['month'].map({12:3,1:3,2:3,3:0,4:0,5:0,6:1,7:1,8:1,9:2,10:2,11:2})
        try:
            import holidays; es=holidays.Spain(years=range(2015,2020))
            df['is_holiday']=t.dt.date.astype(str).map(lambda d:1 if d in [str(h) for h in es] else 0)
        except: df['is_holiday']=0
        return df
    def _cyc(self, df):
        df['hour_sin']=np.sin(2*np.pi*df['hour']/24); df['hour_cos']=np.cos(2*np.pi*df['hour']/24)
        df['dow_sin']=np.sin(2*np.pi*df['dow']/7);   df['dow_cos']=np.cos(2*np.pi*df['dow']/7)
        df['month_sin']=np.sin(2*np.pi*df['month']/12); df['month_cos']=np.cos(2*np.pi*df['month']/12)
        df['doy_sin']=np.sin(2*np.pi*df['day_of_year']/365); df['doy_cos']=np.cos(2*np.pi*df['day_of_year']/365)
        return df
    def _lag(self, df):
        for lag in self.lag_hours: df[f'lag_{lag}h']=df[TARGET].shift(lag)
        return df
    def _roll(self, df):
        s=df[TARGET].shift(1)
        df['rolling_mean_24h']=s.rolling(24,min_periods=12).mean()
        df['rolling_mean_168h']=s.rolling(168,min_periods=84).mean()
        df['rolling_std_24h']=s.rolling(24,min_periods=12).std()
        df['rolling_max_24h']=s.rolling(24,min_periods=12).max()
        df['rolling_min_24h']=s.rolling(24,min_periods=12).min()
        return df
    def _weather(self, df):
        df['temp_squared']=df['temp']**2; df['is_cold']=(df['temp']<280).astype(int)
        df['is_hot']=(df['temp']>298).astype(int); df['temp_humidity']=df['temp']*df['humidity']/100
        df['is_raining']=(df['rain_1h']>0).astype(int); df['wind_chill']=df['wind_speed']*(1-df['temp']/310)
        return df

fe = EnergyFeatureTransformer()
train_fe = fe.fit_transform(train_raw)
X_train  = train_fe[ALL_FEATURES]
y_train  = train_fe[TARGET]
print(f'X_train: {X_train.shape}  |  y_train mean: {y_train.mean():,.0f} MW')

X_train: (24208, 40)  |  y_train mean: 28,563 MW


## 3. Define Optuna Objective Function

The objective function:
1. Receives a `trial` from Optuna
2. Samples hyperparameters using `trial.suggest_*`
3. Cross-validates using `TimeSeriesSplit`
4. Logs the trial as a **nested** MLflow run
5. Returns CV RMSE (Optuna minimizes this)

In [4]:
def make_objective(model_name):
    """Factory: returns the objective function for the given model type."""

    def objective_lgbm(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 200, 1000),
            'max_depth':         trial.suggest_int('max_depth', 3, 10),
            'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
            'random_state':      RANDOM_STATE,
            'verbose':           -1,
            'n_jobs':            -1,
        }
        model    = LGBMRegressor(**params)
        pipeline = Pipeline([('model', model)])
        scores   = cross_val_score(pipeline, X_train, y_train, cv=TSCV,
                                    scoring='neg_root_mean_squared_error')
        cv_rmse  = -scores.mean()
        cv_r2    = cross_val_score(pipeline, X_train, y_train, cv=TSCV, scoring='r2').mean()

        # Log this trial as a NESTED MLflow run
        # nested=True is REQUIRED — we are already inside a parent run
        with mlflow.start_run(run_name=f'trial_{trial.number:03d}', nested=True):
            mlflow.log_params(params)
            mlflow.log_metric('cv_rmse',    cv_rmse)
            mlflow.log_metric('cv_r2',      cv_r2)
            mlflow.log_metric('trial_number', trial.number)

        return cv_rmse  # Optuna minimizes this

    def objective_gbm(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
            'max_depth':         trial.suggest_int('max_depth', 2, 8),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'random_state':      RANDOM_STATE,
        }
        model    = GradientBoostingRegressor(**params)
        pipeline = Pipeline([('model', model)])
        scores   = cross_val_score(pipeline, X_train, y_train, cv=TSCV,
                                    scoring='neg_root_mean_squared_error')
        cv_rmse  = -scores.mean()
        cv_r2    = cross_val_score(pipeline, X_train, y_train, cv=TSCV, scoring='r2').mean()

        with mlflow.start_run(run_name=f'trial_{trial.number:03d}', nested=True):
            mlflow.log_params(params)
            mlflow.log_metric('cv_rmse',      cv_rmse)
            mlflow.log_metric('cv_r2',        cv_r2)
            mlflow.log_metric('trial_number', trial.number)

        return cv_rmse

    if model_name == 'LightGBM' and HAS_LGB:
        return objective_lgbm
    else:
        return objective_gbm

objective = make_objective(BEST_MODEL_NAME)
print(f'Objective function ready for: {BEST_MODEL_NAME}')

Objective function ready for: GradientBoosting


## 4. Run the Study

The study runs inside a **parent MLflow run**.  
Each trial is a nested child run inside the parent.

In [5]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# Parent run — wraps the entire study
with mlflow.start_run(run_name=f'optuna_study_{BEST_MODEL_NAME}'):

    # Log study configuration to parent run
    mlflow.log_param('model_type',  BEST_MODEL_NAME)
    mlflow.log_param('n_trials',    N_TRIALS)
    mlflow.log_param('cv_strategy', 'TimeSeriesSplit n=5 gap=24')
    mlflow.log_param('sampler',     'TPE')
    mlflow.log_param('direction',   'minimize cv_rmse')
    mlflow.log_param('n_features',  len(ALL_FEATURES))

    # Create study
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
    )

    print(f'Running {N_TRIALS} trials for {BEST_MODEL_NAME}...')
    print('Each trial = 5-fold TimeSeriesSplit CV + nested MLflow run')
    print()

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    best_params = study.best_params
    best_rmse   = study.best_value

    # Log best results to parent run
    mlflow.log_metric('best_cv_rmse',    best_rmse)
    mlflow.log_metric('best_trial_num',  study.best_trial.number)
    mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})

    print(f'\n✅ Study complete!')
    print(f'   Best trial:    #{study.best_trial.number}')
    print(f'   Best CV RMSE:  {best_rmse:,.2f} MW')

Running 1 trials for GradientBoosting...
Each trial = 5-fold TimeSeriesSplit CV + nested MLflow run



Best trial: 0. Best value: 593.158: 100%|██████████| 1/1 [01:07<00:00, 67.87s/it]


✅ Study complete!
   Best trial:    #0
   Best CV RMSE:  593.16 MW


## 5. Best Hyperparameters

In [6]:
print('BEST HYPERPARAMETERS:')
print('=' * 45)
for k, v in best_params.items():
    print(f'  {k:30s}: {v}')
print()
print(f'Best CV RMSE: {best_rmse:,.2f} MW')
print()

# Trial statistics
trials_df = study.trials_dataframe()
print(f'Total trials:  {len(trials_df)}')
print(f'Completed:     {(trials_df["state"] == "COMPLETE").sum()}')
print(f'Pruned:        {(trials_df["state"] == "PRUNED").sum()}')
print(f'Failed:        {(trials_df["state"] == "FAIL").sum()}')
print()
print('Top 5 trials by CV RMSE:')
print(trials_df.nsmallest(5, 'value')[['number','value']].rename(
    columns={'number':'trial','value':'cv_rmse'}).to_string(index=False))

BEST HYPERPARAMETERS:
  n_estimators                  : 250
  max_depth                     : 8
  learning_rate                 : 0.1205712628744377
  subsample                     : 0.8394633936788146
  min_samples_split             : 4
  min_samples_leaf              : 2
  max_features                  : log2

Best CV RMSE: 593.16 MW

Total trials:  1
Completed:     1
Pruned:        0
Failed:        0

Top 5 trials by CV RMSE:
 trial    cv_rmse
     0 593.157606


## 6. Optuna Visualizations

In [7]:
# Plot 1: Optimization history — how best score evolved over trials
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(title='Optimization History — Best CV RMSE over Trials')
fig.show()

In [8]:
# Plot 2: Hyperparameter importance — which params drove improvement most
fig = optuna.visualization.plot_param_importances(study)
fig.update_layout(title='Hyperparameter Importance')
fig.show()

ValueError: Cannot evaluate parameter importances with only a single trial.

In [9]:
# Plot 3: Parallel coordinates — visualize all param combos vs score
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.update_layout(title='Parallel Coordinates — Parameter Combinations vs CV RMSE')
fig.show()

In [10]:
# Plot 4: Slice plot — score vs each individual parameter
fig = optuna.visualization.plot_slice(study)
fig.update_layout(title='Slice Plot — CV RMSE vs Each Parameter')
fig.show()

## 7. Save Best Params

In [11]:
# Add fixed params that Optuna doesn't tune
best_params_full = {
    **best_params,
    'random_state': RANDOM_STATE,
}

# LightGBM-specific additions
if BEST_MODEL_NAME == 'LightGBM':
    best_params_full.update({'verbose': -1, 'n_jobs': -1})

with open(PROCESSED_PATH + 'best_params.json', 'w') as f:
    json.dump(best_params_full, f, indent=2)

print('✅ Best params saved to data/processed/best_params.json')
print()
print('Ready for notebook 07 — Final Evaluation.')

✅ Best params saved to data/processed/best_params.json

Ready for notebook 07 — Final Evaluation.
